# Transcript Explorer

## Requirements and assumptions

## Upload your transcripts

Upload your transcripts as a single zip file into the panel on the left.

TODO: add screenshot/s

# Extract Transcript Lines

This step will aim to extract each turn of each uploaded transcript, and separate
the speaker from the turn content. This step will only be as consistent as your
transcripts are.

This will completely delete and recreate the state of processed conversations - if
you want to upload new or edited files, make the contents of the conversations folder
match what you want, and re-run this cell.

In [1]:
# Start by setting up a small database to hold the processed information.

import sqlite3

convo_db_path = "transcripts.db"

convo_db = sqlite3.connect(convo_db_path, isolation_level=None)

convo_db.executescript(
    """
    DROP table if exists turn;
    CREATE table turn (
        turn_id integer primary key,
        source_file,
        turn_no,
        speaker,
        turn,
        unique(source_file, turn_no)
    )
    """
)

# Then we'll identify and load conversation turns from each transcript.
import glob
from pathlib import Path
from zipfile import ZipFile

from docx import Document

convo_db.execute("begin")

uploaded_path = Path("uploaded")

zip_files = glob.glob(str(uploaded_path / "*zip"))

if len(zip_files) > 1:
    raise ValueError(
        "Please upload just one zipfile - "
        "you can remove the ones you don't want by right clicking on "
        "the left hand file browser menu and selecting 'delete'"
    )
elif len(zip_files) == 0:
    raise ValueError("Please upload a zipfile in the left hand file browser.")

zip_file = zip_files[0]

with ZipFile(zip_file) as zipf:
    for name in zipf.namelist():
        # Only process docx files
        if not name.lower().endswith(".docx"):
            continue

        print("Processing", name)
        with zipf.open(name, "r") as transcript_file:
            # Load the word document
            doc = Document(transcript_file)
    
            # We're going to treat each paragraph in the word file as a turn
            for turn_no, paragraph in enumerate(doc.paragraphs):
                para_text = paragraph.text
    
                # Identify speakers by looking for the colon character then a tab.
                # If colon-tab is not matched, no speaker will be assigned to this turn.
                speaker_split = para_text.split(":\t")
    
                if len(speaker_split) == 2:
                    speaker, text = speaker_split
                else:
                    speaker, text = None, para_text
    
                convo_db.execute(
                    "INSERT into turn values (?, ?, ?, ?, ?)",
                    (None, name, turn_no, speaker, text),
                )

convo_db.execute("commit")

Processing Phase 2 Data analysis/Principal Interview Paper Ideas.docx
Processing Phase 2 Data analysis/DP22 School Community Partnerships project case study data summary.docx
Processing Phase 2a Principal Interview/Principal Interview protocol and contacts/FINAL Principal Interview Protocol_250523.docx
Processing Phase 2a Principal Interview/Principal Interview protocol and contacts/School-Community Partnerships Interview Questions.docx
Processing Phase 2a Principal Interview/Principal Interviews Transcriptions/Charters Towers SHS/Principal Interview- Charters Towers SHS 271023.docx
Processing Phase 2a Principal Interview/Principal Interviews Transcriptions/Charters Towers SHS/School-Community Partnerships Chat_CHARTERS TOWERS_2023-10-26.docx
Processing Phase 2a Principal Interview/Principal Interviews Transcriptions/Bowen SHS/Bowen SHS Transcription_FINAL.docx
Processing Phase 2a Principal Interview/Principal Interviews Transcriptions/Bowen SHS/Bowen Transcript.docx
Processing Phase 2


The previous step created a little database holding the extracted turns from all
uploaded files. Now we're going to index this dataset in a way that is aware of the
within and across-turn structure of conversations.


First lets define two functions for breaking down turns into word-like units
(tokenisation). We'll create two tokenisers - the first one normalises case by
lowercasing all of the text, then finds and splits the turn up at characters
indicating word boundaries('\b'), or whitespace characters like space, newlines, and
tabs.

The second, display_tokenise, breaks on the same places, but does not lowercase, or
remove spaces, so we can recreate and highlight search result matches/concordances
of search terms.

In [2]:

import re  # Python's inbuilt regular expression library

boundary_regex = re.compile(r"(\b|\s+)")


def tokenise(text):
    """
    Break down text of posts into words.

    This is a simple tokeniser that breaks on whitespace and simple word 'boundaries'
    based on character classes (for example, at punctuation).

    """
    return [
        stripped
        for token in boundary_regex.split(text.lower())
        if (stripped := token.strip())
    ]


def display_tokenise(text):
    """Tokenise, but preserving punctuation and whitespace for display as original."""
    text_length = len(text)

    # Find split matches, padding with a split at the start/end - if it's not needed
    # they will be merged with existing matches.
    matches = [
        (0, 0),
        *[m.span() for m in boundary_regex.finditer(text)],
        (text_length, text_length),
    ]
    merged = [matches[0]]

    for start, end in matches[1:]:

        last_start, last_end = merged[-1]

        if start == last_end:
            merged[-1] = (last_start, end)
        else:
            merged.append((start, end))

    token_starts = [m[1] for m in merged]

    return [text[start:end] for start, end in zip(token_starts, token_starts[1:])]

# Index the transcripts

Now we're going to make the transcripts searchable using words by indexing with the
the hyperreal library. This involves constructing a corpus that describes what we want
to consider a 'document' for the purposes of retrieval, display and search.

This indexing process is aware of basic conversational structure however: instead of
indexing just the text of the individual turn, we break things down by considering:

1. Whether a token occurs in position initial, final, or medial.
2. Whether a token occurs in the first, second, or third turn. This means that each
   turn is indexed three times, in the three relative positions, to make it easier to
   construct queries that are relative to this structure.

In [3]:
from tinyhtml import h

from hyperreal import field_types
from hyperreal.corpus import HyperrealCorpus


class TranscriptCorpus(HyperrealCorpus):

    @property
    def db(self):
        if not hasattr(self, "_db"):
            self._db = sqlite3.connect(convo_db_path, isolation_level=None)
        return self._db

    def __getstate__(self):
        return

    def __setstate__(self, *args):
        self._db = None

    def all_doc_keys(self):
        return (r[0] for r in self.db.execute("select turn_id from turn"))

    def docs(self, doc_keys):

        for key in doc_keys:
            # key points to the current turn as the relative turn 1 -> we also retrieve
            # the following two turns as relative turns 2 and 3.
            turns = self.db.execute(
                """
                WITH rolling_turn as (
                    select 
                        source_file, 
                        turn_no
                    from turn
                    where turn_id = ?
                )
                SELECT 
                    turn_id,
                    source_file,
                    turn_no,
                    turn_no - (select turn_no from rolling_turn) + 1 as turn_offset,
                    coalesce(speaker, '<no speaker>') as speaker,
                    turn
                from turn
                where source_file = (select source_file from rolling_turn)
                    and turn_no 
                        between (select turn_no from rolling_turn) and 
                            (select turn_no + 2 from rolling_turn) 
                """,
                [key],
            )

            names = [col[0] for col in turns.description]

            turn_rows = []

            for turn in turns:
                turn_data = {k: v for k, v in zip(names, turn)}

                source_file = turn_data["source_file"]
                parts = Path(source_file).parts

                turn_data["file"] = parts[-1]
                turn_data["path"] = "/".join(parts[1:-1])

                turn_rows.append(turn_data)

            yield key, turn_rows

    def doc_to_features(self, doc):
        indexed = {}

        for key in ("turn_no", "file", "path", "speaker"):
            indexed[key] = field_types.Value(doc[0][key])

        for turn_data in doc:

            turn_offset = turn_data["turn_offset"]
            turn_field = f"t{turn_offset}"

            turn_tokens = tokenise(turn_data["turn"])

            if turn_tokens:
                pos_initial = turn_tokens[0]
                pos_final = turn_tokens[-1]
                pos_medial = turn_tokens[1:-1]

                indexed[turn_field + ",initial"] = field_types.Value(pos_initial)
                indexed[turn_field + ",final"] = field_types.Value(pos_final)
                indexed[turn_field + ",medial"] = field_types.ValueSequence(pos_medial)
                indexed[turn_field] = field_types.ValueSequence(turn_tokens)

            indexed[turn_field + ",speaker"] = field_types.Value(turn_data["speaker"])

        return indexed

    def doc_to_display_features(self, doc):

        # Note we need to be careful here because turn 2/3 will not exist at the last
        # of the transcript
        display = {}

        for i, turn in enumerate(doc):
            display[f"t{i+1}"] = display_tokenise(doc[i]["turn"])

        return display

    def features_to_html_concordance(
        self, doc_features, display_features, highlight_features
    ):
        """Do not render concordances: turns are short already."""
        return None

    def render_turn(self, turn_no, speaker, turn):
        if speaker:
            return (turn_no, " ", h("em")(speaker), ":\t", turn)
        else:
            return (turn_no, turn)

    def doc_to_html(self, doc, highlight_features=None):

        turn_block = self.db.execute(
            """
            SELECT turn_no, speaker, turn, turn_id - ?1 + 1 as relative_turn
            from turn
            where turn_id between ?1 - 2 and ?1 + 2
            order by turn_id
            """,
            [doc[0]["turn_id"]],
        )

        return h("div")(
            h("h3")(doc[0]["path"], " ", doc[0]["file"]),
            h("ol")(
                h("li", klass=f"turn{turn[-1]}")(self.render_turn(*turn[:3]))
                for turn in turn_block
            ),
        )

    extra_css = """
        .turn1 {
            background-color: lightgray;
        }
    """

    def close(self):
        if hasattr(self, "_db"):
            self._db.close()


The above TranscriptCorpus only describes how to work with this collection of
documents - it doesn't actually do anything yet. Let's make it actually do
something.


In [4]:
from loky import get_reusable_executor

from hyperreal.index_core import HyperrealIndex, TableFilter

pool = get_reusable_executor()

convo_corpus = TranscriptCorpus()

convo_idx = HyperrealIndex(
    "conversations_index.db", convo_corpus, pool
)

convo_idx.rebuild()

In [5]:
clustering = convo_idx.plugins["feature_clusters"]

include_fields = ["t1"]

random_clustering = clustering.initialise_random_clustering(
    include_fields=include_fields, min_docs=3, n_clusters=64
)
clustering.replace_clusters(random_clustering)
refined = clustering.refine_clustering(iterations=50)

0 81.06826734890254 5590 2685 0
1 86.01436931003794 4910 2262 0
2 89.53917230945736 4379 2113 0
3 93.59348549921135 3950 1869 0
4 96.33404069226454 3409 1623 0
5 98.61302983339657 3048 1467 0
6 100.97242875012401 2750 1290 0
7 103.21303743095433 2505 1162 0
8 105.2156377592667 2290 1057 0
9 107.35325079149449 2045 954 0
10 108.63776319320287 1887 890 0
11 109.97251968939462 1772 801 0
12 111.06985656835101 1638 722 0
13 111.99403139316026 1524 690 0
14 113.08495876219853 1453 666 0
15 113.96083248118545 1269 554 0
16 114.58803451260087 1193 536 0
17 115.155966229899 1092 493 0
18 116.4748890430997 997 444 0
19 117.17193361997877 925 415 0
20 117.71089486327672 829 364 0
21 118.6309731298461 768 341 0
22 118.90263958838217 736 311 0
23 119.2696535934677 711 308 0
24 119.59755181689094 639 273 0
25 120.68738627438108 592 242 0
26 121.49951353048247 586 244 0
27 121.67627522383705 529 218 0
28 121.8656853047374 512 216 0
29 122.21675615535729 482 214 0
30 122.37771235988427 457 194 0
31 1

In [6]:
import asyncio
import os

from hyperreal.web_server import serve_index

print("launching web server")

search_fields = include_fields + ["speaker"]

for i in range(1, 4):
    for subfield in ("initial", "final", "medial"):
        search_fields.append(f"t{i},{subfield}")

convo_idx.facets = [
    (
        "Speaker",
        convo_idx.field_features("speaker", min_docs=1),
        TableFilter(order_by="hits", first_k=20, keep_above=0),
    ),
    (
        "File",
        convo_idx.field_features("file"),
        TableFilter(order_by="hits", first_k=20, keep_above=0),
    ),
    (
        "Path",
        convo_idx.field_features("path"),
        TableFilter(order_by="hits", first_k=20, keep_above=0),
    ),
]

convo_idx.search_fields = {field: tokenise for field in search_fields}

jupyter_hub_service_prefix = os.environ.get("JUPYTERHUB_SERVICE_PREFIX", "/")
url_base = jupyter_hub_service_prefix + "proxy/absolute/9999"

loop = asyncio.get_running_loop()
task = loop.create_task(serve_index(convo_idx, base_path=url_base))

display(h("a", href=url_base + "/browse/")("Browse your transcripts"))


launching web server


raw('<a href="/proxy/absolute/9999/browse/">Browse your conversations here</a>')